# Part 1 · Where do people want to go? — OD weights and flow propagation

In this notebook you change **who wants to go where** in a small pedestrian network and watch
how that changes *where* and *when* congestion appears.

**By the end you should be able to explain:**
1. what an OD weight is and why it is *relative*,
2. why the *mix* of OD pairs matters, not only the totals,
3. how a change in destinations over time travels through the network,
4. how route choice can soften (or not) the effect of a bad OD pattern.

Work in pairs: one person drives the keyboard, the other writes down predictions. Swap every exercise.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# On Google Colab this installs PedNStream and fetches the workshop files.
# On your own laptop, follow README.md instead and this cell does nothing.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("workshop_tools.py"):
    WORKSHOP_REPO = "https://github.com/WaimenMak/pednstream-workshop.git"  # <- facilitator: set this
    !git clone -q {WORKSHOP_REPO} ws && cp -r ws/* .
    %pip install -q -r requirements.txt

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 90
from workshop_tools import run, compare, compare_maps, switch, ramp
print("Ready ✔")

## 0 · Meet *Station Square*

```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```

* **Two entrances** (green squares): West Gate **A** enters at the Plaza, South Gate **B** enters at the Hall.
* **Two destinations** (red diamonds): the **Platform X** and the **City Exit Y**.
* Corridor **2_3** (Plaza–Hall) is only **2 m** wide: the obvious suspect for trouble.
* Each gate receives a rush of people that peaks around minute 17. The rush is the **same in every scenario**.

### What is an OD weight?
`od={"0_5": 3, "0_6": 1}` does **not** mean "3 people". It means: *of the people entering at gate 0,
3 out of every 3 + 1 = 4 want destination 5*. How many people enter is set by the gate's **demand**.
The OD weights only decide **how that demand is split** over destinations.

> **Quick check:** `{"0_5": 10, "0_6": 10}` and `{"0_5": 1, "0_6": 1}`: same or different result? Why?

In [ ]:
base = run()   # all four OD weights = 1  ->  every gate sends 50% to X and 50% to Y
base.dashboard("Baseline: everyone 50/50");

In [ ]:
base.kpis()

**How to read the outputs**
* **Map** (left): corridor colour = density at the most crowded moment. Above **2 ped/m²** (the
  dashed line on the right) a corridor is *congested*: walking speed drops sharply.
* **Density per corridor**: when and where congestion builds up and clears.
* **Arrivals**: cumulative people reaching each destination. *Turned away* = people who could not even
  enter because the first corridor was full.
* **time spent inside (person-hours)**: our main "how bad was it" number. Lower is better.

---
## Exercise 1 · Everyone to one destination
Imagine a train is delayed and **everyone** heads to the Platform X. Then imagine the opposite: a
concert in town, and **everyone** heads to the City Exit Y.

> ✋ **Predict before you run.** Which corridor will be the worst in each case? Which case will be worse overall?
>
> Write your guess here (double-click to edit): …

In [ ]:
all_x = run(od={"0_5": 1, "0_6": 0, "1_5": 1, "1_6": 0})
all_y = run(od={"0_5": 0, "0_6": 1, "1_5": 0, "1_6": 1})

compare({"baseline": base, "all to X": all_x, "all to Y": all_y})

In [ ]:
compare_maps({"baseline": base, "all to X": all_x, "all to Y": all_y});

**Discuss**
1. In *all to X*, people are turned away at Gate B. Why B and not A? (Hint: how many routes does B have to X?)
2. The narrow corridor 2_3 is *not* always the worst one. Where did the congestion go instead, and why?
   Look at `all_x.route_split()`.

In [ ]:
all_x.route_split()

---
## Exercise 2 · Same totals, different mix ⭐
Both scenarios below send **exactly half** of all travellers to X and half to Y. Only *who* goes where differs:

| scenario | Gate A goes to | Gate B goes to |
|---|---|---|
| **crossing** | Platform X (east) | City Exit Y (north) |
| **parallel** | City Exit Y (north) | Platform X (east) |

> ✋ **Predict before you run.** Will the two scenarios give (almost) the same result? If not, which is worse, and where?
>
> Write your guess here (double-click to edit): …

In [ ]:
crossing = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1})
parallel = run(od={"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0})

compare({"baseline": base, "crossing": crossing, "parallel": parallel})

In [ ]:
compare_maps({"crossing": crossing, "parallel": parallel});

In [ ]:
crossing.link_table()

**Discuss**
1. Destination totals are identical, yet one scenario costs much more person-time. Explain it in one
   sentence, using the words *route overlap* or *counter-flow*.
2. Which corridors carry people walking in **both directions** in the crossing case?
3. What does this mean for someone who only knows "50% of people go to the Platform"? Is that
   enough information to predict congestion?

---
## Exercise 3 · Destinations that change over time
At **step 150 (minute 25)** a train is announced. From then on, people arriving at both gates are
**6× more likely** to head to the Platform. `switch(before, after, at_step)` builds a weight that changes over time.

> ✋ **Predict before you run.** People already walking towards the City Exit at minute 25: do they keep going, or do they also turn towards the Platform? Which corridor do you expect to react first?
>
> Write your guess here (double-click to edit): …

In [ ]:
announce = run(od={"0_5": switch(1, 6, at_step=150), "0_6": 1,
                    "1_5": switch(1, 6, at_step=150), "1_6": 1})
announce.dashboard("Train announced at minute 25");

In [ ]:
announce.plot_links(["0_2", "2_3", "3_5", "2_4", "4_6"]);

**Discuss**
1. Zoom in on minute 25. Do the corridors react with a travel-time delay, or all at once?
2. PedNStream applies the **current** OD weights at every junction, so people already inside the
   network "re-decide" at the next junction. Is that realistic for a PA announcement? For a
   departure board only visible at the entrance? How would you model the second case?
3. Try the announcement at the *peak* instead: `at_step=105`. Does the *timing* of a change in OD
   weights matter as much as its *size*?

In [ ]:
# your turn: announcement at the peak

---
## Exercise 4 · Do people dodge the jam?
In PedNStream people pick between their two routes using a utility that includes **distance**,
**congestion** and **width**. The congestion weight is `beta`: `beta=0` means people ignore crowding,
a large `beta` means people strongly avoid crowded corridors.

> ✋ **Predict before you run.** For the bad *crossing* pattern from Exercise 2, will raising beta remove the congestion?
>
> Write your guess here (double-click to edit): …

In [ ]:
ignore = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1}, route_choice={"beta": 0})
avoid  = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1}, route_choice={"beta": 8})

compare({"beta = 0 (ignore crowds)": ignore, "beta = 2 (default)": crossing, "beta = 8 (avoid crowds)": avoid})

In [ ]:
avoid.route_split()

**Discuss**
1. Route choice changes the *peak density* more than the *total time spent*. Why can't smart walkers
   fix a bad OD pattern on their own? (Hint: look at what the alternative routes have in common.)
2. Compare with an infrastructure fix: `run(od=..., links={"2_3": {"width": 4}})`. Which helps more
   here, better behaviour or a wider corridor?

In [ ]:
# your turn: widen corridor 2_3 in the crossing scenario

---
## Wrap-up: write down three sentences
1. An OD weight controls …
2. Two OD patterns with the same destination totals can behave differently because …
3. A change in OD weights reaches a corridor *X* minutes later because …

➡ Next: **`02_playground.ipynb`** (free exploration with sliders), then **`03_challenge.ipynb`**.